#### Feature Encoding

AIM: To perform feature encoding on the labeled FRA Atlas dataset (labeled_dataset.csv).


1) Imports And Loading Dataset

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
df = pd.read_csv("../ml/data/labeled_dataset.csv")
df.head()

,plot_id,state,district,lat,lon,land_area_ha,claim_type,soil_ph,soil_organic_carbon,soil_clay_pct,...,soil_ph_imputed,soil_organic_carbon_imputed,soil_clay_pct_imputed,elevation_m_imputed,pm_kisan_score,mgnrega_score,jjm_score,pmfby_score,pmksy_score,soil_health_card_score
0,SYN-00001,Madhya Pradesh,Chhindwara,22.03435,78.92785,2.63,IFR,7.1,17.22,43.96,...,True,True,True,False,84.28,53.77,63.18,59.21,60.99,11.51
1,SYN-00002,Madhya Pradesh,Chhindwara,22.04054,78.91853,1.25,IFR,7.4,14.40,46.00,...,False,False,False,False,86.66,53.30,62.54,55.70,67.43,22.15
2,SYN-00003,Madhya Pradesh,Chhindwara,22.06489,78.91472,3.00,CR,7.4,14.40,45.10,...,False,False,False,True,25.28,50.69,59.42,48.91,64.49,22.74
3,SYN-00004,Madhya Pradesh,Chhindwara,22.04481,78.94029,3.59,IFR,7.1,17.22,43.96,...,True,True,True,False,85.21,54.10,67.04,46.28,60.25,19.46
4,SYN-00005,Madhya Pradesh,Chhindwara,22.06331,78.95167,1.80,IFR,7.1,17.22,43.96,...,True,True,True,True,89.37,53.43,66.11,56.49,69.63,13.05


2) Label Encoding: Used to convert categorical values into numerical labels. Here, 'claim_type' is label encoded.

In [3]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["claim_type_label"] = label_encoder.fit_transform(df["claim_type"])

df[["claim_type", "claim_type_label"]].head(10)

,claim_type,claim_type_label
0,IFR,2
1,IFR,2
2,CR,1
3,IFR,2
4,IFR,2
5,IFR,2
6,IFR,2
7,IFR,2
8,CR,1
9,IFR,2


Observation: 
Label Encoding assigns numerical values such as 0, 1, and 2 to categories.
This can create an artificial ordering between categories.

For example, if CR = 0, IFR = 1, and OTH = 2, a model may incorrectly
interpret OTH as greater than IFR and IFR as greater than CR.

Since `claim_type` is a nominal categorical feature with no natural order,
One-Hot Encoding is more appropriate.

3) One-Hot Encoding

In [4]:
claim_type_encoded = pd.get_dummies(
    df["claim_type"],
    prefix="claim_type"
)

claim_type_encoded.head()

,claim_type_CFR,claim_type_CR,claim_type_IFR
0,False,False,True
1,False,False,True
2,False,True,False
3,False,False,True
4,False,False,True


One-Hot Encoding on State/District: 
1. The `state` and `district` columns are also categorical features.
2. They are converted into separate binary columns using One-Hot Encoding.

In [5]:
location_encoded = pd.get_dummies(
    df[["state", "district"]],
    columns=["state", "district"],
    prefix=["state", "district"]
)

location_encoded.head()

,state_Madhya Pradesh,state_Odisha,state_Telangana,state_Tripura,district_Adilabad,district_Chhindwara,district_Kandhamal,district_Koraput,district_Shivpuri,district_Warangal,district_West Tripura
0,True,False,False,False,False,True,False,False,False,False,False
1,True,False,False,False,False,True,False,False,False,False,False
2,True,False,False,False,False,True,False,False,False,False,False
3,True,False,False,False,False,True,False,False,False,False,False
4,True,False,False,False,False,True,False,False,False,False,False


4) Encoding the Boolean Column: The `is_scheduled_area` column contains True/False values.



In [6]:
df["is_scheduled_area"] = df["is_scheduled_area"].astype(int)

df[["is_scheduled_area"]].head(10)

,is_scheduled_area
0,1
1,1
2,1
3,1
4,1
5,1
6,1
7,1
8,1
9,1


5) Before/After Comparison Table

In [7]:
comparison = df[[
    "claim_type",
    "state",
    "district"
]].head(5).copy()

comparison

,claim_type,state,district
0,IFR,Madhya Pradesh,Chhindwara
1,IFR,Madhya Pradesh,Chhindwara
2,CR,Madhya Pradesh,Chhindwara
3,IFR,Madhya Pradesh,Chhindwara
4,IFR,Madhya Pradesh,Chhindwara


In [8]:
encoded_comparison = pd.concat([
    claim_type_encoded.head(5).reset_index(drop=True),
    location_encoded.head(5).reset_index(drop=True),
    df[["is_scheduled_area"]].head(5).reset_index(drop=True)
], axis=1)

encoded_comparison

,claim_type_CFR,claim_type_CR,claim_type_IFR,state_Madhya Pradesh,state_Odisha,state_Telangana,state_Tripura,district_Adilabad,district_Chhindwara,district_Kandhamal,district_Koraput,district_Shivpuri,district_Warangal,district_West Tripura,is_scheduled_area
0,False,False,True,True,False,False,False,False,True,False,False,False,False,False,1
1,False,False,True,True,False,False,False,False,True,False,False,False,False,False,1
2,False,True,False,True,False,False,False,False,True,False,False,False,False,False,1
3,False,False,True,True,False,False,False,False,True,False,False,False,False,False,1
4,False,False,True,True,False,False,False,False,True,False,False,False,False,False,1


CONCLUSION: The original categorical features are represented by numerical/binary
features after encoding. One-Hot Encoding avoids imposing an artificial
order on `claim_type`, `state`, and `district`, while the Boolean feature
is represented as 0 and 1.